In [69]:
# RNN 
# 재귀적 구조 (Recurrence) 사용, 한 번에 한 단어(타임 스탭)씩 순서대로 읽음
# 읽은 내용을 은닉 상태(hidden state)라는 '기억'에 쌓음

In [70]:
# RNN이 지원하는 5가지 입출력 구조
# One-to_One            : 고정 입력 → 고정 출력
# One-to-Many           : 고정 입력 → 시퀀스 출력
# Many-to-One           : 시퀀스 입력 → 고정 출력
# Many-to-Many (지연)    : 시퀀스 → 시퀀스 (순서 존재)
# Many-to-Many (동기)    : 시퀀스 → 시퀀스 (동시)

In [71]:
# Vanilla 
# ht​=tanh(Whh​⋅ht−1​+Wxh​⋅xt​)
# yt​=Why​⋅ht​
# xt​   : 현재 타임스텝의 입력 벡터ht−1h_{t-1}
# ht−1 : ​이전 타임스텝의 은닉 상태 (기억)hth_t
# ht​   : 현재 은닉 상태 (새로운 기억)WxhW_{xh}
# Wxh​  : 입력 → 은닉 가중치WhhW_{hh}
# Whh​  : 은닉 → 은닉 가중치 (기억 전달)WhyW_{hy}
# Why  : ​은닉 → 출력 가중치tanh⁡\tanh
# tanh : 활성화 함수 (-1 ~ 1 사이로 압축)

In [72]:
# 기울기 소실 문제 (Vanishing Gradient)
# 역전파(BPTT)를 할 때, 과거로 갈수록 기울기가 계속 Whh 와 tanh 를 곱하면서 전달
# tanh <= 1 -> 타임스탭이 길어질수록 기울기가 0에 수렴

In [73]:
# "hello" 예제
# 1. 데이터 준비
# 2. One-Hot 인코딩
# 3. 시퀀스 생성
# 4. 모델 정의
# 5. 모델 초기화
# 6. 학습
# 7. 결과 확인

In [74]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 기본 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

np.random.seed(42)
torch.manual_seed(42)

Device: cpu


In [75]:
# ────────────────────────────────────────
# 1. 데이터 준비
# ────────────────────────────────────────
text = "hello"
chars = sorted(list(set(text)))       # ['e', 'h', 'l', 'o']
vocab_size = len(chars)               # 4

char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

print(f"어휘: {chars}")
print(f"문자→인덱스: {char_to_idx}")

어휘: ['e', 'h', 'l', 'o']
문자→인덱스: {'e': 0, 'h': 1, 'l': 2, 'o': 3}


In [76]:
# ────────────────────────────────────────
# 2. One-Hot 인코딩
# ────────────────────────────────────────
def one_hot_encode(char_idx, vocab_size):
    one_hot = np.zeros(vocab_size)
    one_hot[char_idx] = 1
    return one_hot

In [77]:
# ────────────────────────────────────────
# 3. 시퀀스 데이터 생성
# ────────────────────────────────────────
X_indices = [char_to_idx[c] for c in text[:-1]]   # h, e, l, l
Y_indices = [char_to_idx[c] for c in text[1:]]    # e, l, l, o

print("\n입력-타겟 쌍:")
for i, (xi, yi) in enumerate(zip(X_indices, Y_indices)):
    print(f"  Step {i+1}: '{idx_to_char[xi]}' → '{idx_to_char[yi]}'")

X_one_hot = np.array([one_hot_encode(i, vocab_size) for i in X_indices])
Y_one_hot = np.array([one_hot_encode(i, vocab_size) for i in Y_indices])

X = torch.tensor(X_one_hot, dtype=torch.float32, device=device)
Y = torch.tensor(Y_one_hot, dtype=torch.float32, device=device)

print(f"\nX shape: {X.shape}, Y shape: {Y.shape}")



입력-타겟 쌍:
  Step 1: 'h' → 'e'
  Step 2: 'e' → 'l'
  Step 3: 'l' → 'l'
  Step 4: 'l' → 'o'

X shape: torch.Size([4, 4]), Y shape: torch.Size([4, 4])


In [78]:
# ────────────────────────────────────────
# 4. 모델 정의
# ────────────────────────────────────────
class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size)  * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.W_hy = nn.Parameter(torch.randn(output_size, hidden_size) * 0.01)
        self.b_h  = nn.Parameter(torch.zeros(hidden_size, 1))
        self.b_y  = nn.Parameter(torch.zeros(output_size, 1))

    def forward(self, x_sequence):
        seq_len = x_sequence.shape[0]
        h_t = torch.zeros(self.hidden_size, 1, device=device)

        y_preds      = []
        hidden_states = [h_t.clone()]

        for t in range(seq_len):
            x_t = x_sequence[t].unsqueeze(1)                          # (4,) → (4,1)
            h_t = torch.tanh(self.W_hh @ h_t + self.W_xh @ x_t + self.b_h)
            y_t = self.W_hy @ h_t + self.b_y
            y_preds.append(y_t.squeeze(1))
            hidden_states.append(h_t.clone())

        return torch.stack(y_preds), torch.stack(hidden_states)

In [79]:
# ────────────────────────────────────────
# 5. 모델 초기화
# ────────────────────────────────────────
input_size  = vocab_size   # 4
hidden_size = 3
output_size = vocab_size   # 4
learning_rate = 0.1
num_epochs = 100

model     = VanillaRNN(input_size, hidden_size, output_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("\n모델 파라미터:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")



모델 파라미터:
  W_xh: torch.Size([3, 4])
  W_hh: torch.Size([3, 3])
  W_hy: torch.Size([4, 3])
  b_h: torch.Size([3, 1])
  b_y: torch.Size([4, 1])


In [80]:
# ────────────────────────────────────────
# 6. 학습
# ────────────────────────────────────────
losses = []
print("\n학습 시작...")
print(f"{'Epoch':<10} {'Loss':<15} {'Accuracy'}")
print("-" * 40)

for epoch in range(num_epochs):
    model.train()
    y_preds, hidden_states = model(X)
    loss = criterion(y_preds, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    preds   = y_preds.argmax(dim=1)
    targets = Y.argmax(dim=1)
    acc = (preds == targets).float().mean().item()

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"{epoch+1:<10} {loss.item():<15.6f} {acc:.2%}")

print("-" * 40)
print("학습 완료!")


학습 시작...
Epoch      Loss            Accuracy
----------------------------------------
1          1.386269        50.00%
20         0.036312        100.00%
40         0.001905        100.00%
60         0.001088        100.00%
80         0.000890        100.00%
100        0.000775        100.00%
----------------------------------------
학습 완료!


In [81]:
# ────────────────────────────────────────
# 7. 최종 결과 확인
# ────────────────────────────────────────
model.eval()
with torch.no_grad():
    y_preds, hidden_states = model(X)

predictions = y_preds.argmax(dim=1)
targets     = Y.argmax(dim=1)
final_acc   = (predictions == targets).float().mean().item()

print("\n최종 예측 결과:")
for i, (pred, target) in enumerate(zip(predictions, targets)):
    pred_char   = idx_to_char[pred.item()]
    target_char = idx_to_char[target.item()]
    input_char  = idx_to_char[X_indices[i]]
    mark = "✓" if pred == target else "✗"
    print(f"  Step {i+1}: '{input_char}' → 예측: '{pred_char}'  정답: '{target_char}'  {mark}")

print(f"\n최종 정확도: {final_acc:.2%}")


최종 예측 결과:
  Step 1: 'h' → 예측: 'e'  정답: 'e'  ✓
  Step 2: 'e' → 예측: 'l'  정답: 'l'  ✓
  Step 3: 'l' → 예측: 'l'  정답: 'l'  ✓
  Step 4: 'l' → 예측: 'o'  정답: 'o'  ✓

최종 정확도: 100.00%


In [82]:
# ────────────────────────────────────────
# 8. 타임스텝별 상세 분석
# ────────────────────────────────────────
print("\n타임스텝별 상세 분석:")
for t in range(len(text) - 1):
    print(f"\n{'='*50}")
    print(f"t={t+1}: 입력 '{text[t]}'")
    x_t = X[t].cpu().numpy()
    h_t = hidden_states[t+1].squeeze().detach().cpu().numpy()
    y_t = y_preds[t].detach().cpu().numpy()
    probs = nn.functional.softmax(torch.tensor(y_t), dim=0).numpy()
    pred_idx   = np.argmax(y_t)
    target_idx = Y_indices[t]

    print(f"  입력 벡터:     {x_t}")
    print(f"  은닉 상태 h_t: {h_t}")
    print(f"  출력 로짓 y_t: {y_t}")
    print(f"  softmax 확률:  {probs}")
    print(f"  예측: '{idx_to_char[pred_idx]}'  정답: '{idx_to_char[target_idx]}'  {'✓' if pred_idx == target_idx else '✗'}")


타임스텝별 상세 분석:

t=1: 입력 'h'
  입력 벡터:     [0. 1. 0. 0.]
  은닉 상태 h_t: [ 0.98825014 -0.98274755 -0.9695919 ]
  출력 로짓 y_t: [ 6.80402   -1.0815194 -0.6583415 -1.7730242]
  softmax 확률:  [9.9886250e-01 3.7571593e-04 5.7364517e-04 1.8816664e-04]
  예측: 'e'  정답: 'e'  ✓

t=2: 입력 'e'
  입력 벡터:     [1. 0. 0. 0.]
  은닉 상태 h_t: [0.9993035  0.07429996 0.99999183]
  출력 로짓 y_t: [-0.73222935 -3.8326058   6.5659394  -4.3027596 ]
  softmax 확률:  [6.7628582e-04 3.0454667e-05 9.9927419e-01 1.9031309e-05]
  예측: 'l'  정답: 'l'  ✓

t=3: 입력 'l'
  입력 벡터:     [0. 0. 1. 0.]
  은닉 상태 h_t: [0.666021   0.99972343 0.8932384 ]
  출력 로짓 y_t: [-3.1363554 -3.4789262  6.971521  -1.1402913]
  softmax 확률:  [4.0742230e-05 2.8924640e-05 9.9963045e-01 2.9986378e-04]
  예측: 'l'  정답: 'l'  ✓

t=4: 입력 'l'
  입력 벡터:     [0. 0. 1. 0.]
  은닉 상태 h_t: [-0.97552013  0.9991175  -0.8982757 ]
  출력 로짓 y_t: [-0.6672319 -0.0944345 -1.305777   7.599816 ]
  softmax 확률:  [2.5662480e-04 4.5505253e-04 1.3551334e-04 9.9915278e-01]
  예측: 'o'  정답: 'o'  ✓


## GRU

In [83]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import re
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

In [84]:
# 데이터 불러오기
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
sample_df = df.iloc[-1000:]     # 마지막 1000개 행만

In [85]:
# 정답 레이블 만들기
sample_df['target'] = sample_df['rating'].apply(lambda x : 1 if x > 0.5 else 0)

In [86]:
# 텍스트 전처리 (한글만 남기기)
import re
def clean_text(text):
    return re.sub(r'[가-힣\s]', '', text).strip()

df['clean'] = df['review'].apply(lambda x : clean_text(x))
# 모든 리뷰에 clean_text 함수를 적용

In [87]:
#from konlpy.tag import Okt
# 토큰화 (단어 쪼개기)
df['tokens'] = df['clean'].apply(lambda x : x.split())

In [88]:
# 단어 사전(vocab) 만들기
vocab_size = 10000
counter = Counter()
for tokens in df['tokens']:
    counter.update(tokens)

vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1,
}
for word, freq in counter.most_common(vocab_size):  # 가장 많이 나온 순서대로 최대 10000개
    if len(word) >= 2:
        vocab[word] = len(vocab)
        # 단어를 하나 추가 할 때마다 vocab의 크기가 1씩 커짐
    
print(f"단어수 : {len(vocab)}")

단어수 : 4625


In [89]:
# 리뷰를 숫자 시퀀스로 변환 + 패딩
max_len = 50

def encode(tokens):
    return [vocab.get(t, 1) for t in tokens]

def pad_sequence(seq, max_len=50):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))


X = torch.LongTensor([pad_sequence(encode(t)) for t in df['tokens']])
y = df['rating'].apply(lambda x: x > 0.5).astype(int).values
y = torch.FloatTensor(y)

In [90]:
# 학습/테스트 데이터 분리 + DataLoader 만들기
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)
x_train_dataset = TensorDataset(x_train, y_train)   
x_train_dataloader = DataLoader(x_train_dataset, batch_size=32, shuffle=True)

# TensorDataset(x, y) : 입력 데이터 x와 정답y를 하나로 묶어줌 => 학습할 때 x, y가 쌍으로 나와야함
# DataLoader(datset, batch_size, shuffle) : 데이터를 batch_size개씩 묶어서 학습에 넣어줌
# - shuffle : 학습할 때마다 데이터 순서 섞음 => 순서 외우는거 방지 (테스트 용은 필요 X)

x_test_dataset = TensorDataset(x_test, y_test)
x_test_dataloader = DataLoader(x_test_dataset, batch_size=32)

In [91]:
# GRU 모델 정의
class GRUClassification(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(     # 숫자로 된 단어를 의미있는 벡터로 바꿔줌
            num_embeddings = vocab_size,   # 사전에 있는 단어 수만큼 벡터 만듬
            embedding_dim = embed_dim,     # 각 단어를 dmbed_dim차원 벡터로 표현
            padding_idx = 0                # <PAD>(0번)는 학습에서 제외
        )
        self.gru = nn.GRU(
            input_size = embed_dim,        # 입력은 Embedding에서 나온 embed_dim
            hidden_size = hidden_dim,      # 은닉 상태를 hidden_dim차원으로 압축
            batch_first = True             # 입력 데이터의 첫 번째 차원이 배치(32개)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)              # 숫자 시퀀스를 벡터로 변환
        output, hidden = self.gru(x)       # output 모든 타임 스텝의 은닉상태, hidden 마지막 타임 스텝의 은닉 상태
        x = self.fc(hidden[-1])            
        return x 

In [92]:
# 모델 초기화
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GRUClassification(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [93]:
# 학습루프
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in x_train_dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch).squeeze(-1)
        loss = criterion(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch:{epoch+1} Loss:{total_loss/len(x_train_dataloader):.4f}")

Epoch:1 Loss:0.2093
Epoch:2 Loss:0.1816
Epoch:3 Loss:0.1808
Epoch:4 Loss:0.1828
Epoch:5 Loss:0.1828


In [94]:
# 예측 함수
def predict(text):
    text = clean_text(text)
    tokens = text.split()
    encoded = [vocab.get(t, 1) for t in tokens]
    padded = pad_sequence(encoded)
    tensor = torch.LongTensor([padded]).to(device)
    model.eval()
    with torch.no_grad():
        pred = model(tensor).squeeze(-1)
    score = torch.sigmoid(pred).item()
    if score >= 0.5:
        print("긍정")
    else:
        print("부정")

In [95]:
predict("이 영화 진짜 재미있다 최고")
predict("너무 별로야 시간 낭비")
predict("오랜만에 영화관에서 본 영화인데 좀 시간낭비...?")

긍정
긍정
긍정


In [96]:
print(sample_df['target'].value_counts())

target
1    991
0      9
Name: count, dtype: int64
